In [ ]:
# import
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings
import pandas as pd
import torch
import umap
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import Image

# Our Modules
from s0_config import ISOFORMS, TARGET_COLS, GROUP_COL, SMILES_COL, UNDERPOWERED_N, CLASS_I_ISOFORMS
from s1_split import splitter, generate_fingerprints, fps_to_numpy, nn_median, draw_split
from s2_metrics import (calculate_metrics, per_isoform_report, parity_plots, ranking_metrics_per_isoform,
                        r2_comparison_plot, full_per_isoform_table, INK, INK_SOFT, GRID, SURFACE,
                        C_BLUE, C_ORANGE)
from s3_baseline import run_lgbm_baseline
from s5_training import train_dmpnn, select_best_seed, seed_view, predict_dmpnn
from s6_stacking import fit_stacking_meta_learner, predict_stacking
from s7_inference import export_bundle, load_bundle, predict, predict_to_dataframe
from s8_screening import selectivity_score, largest_fragment, save_top_table, ecdf, summ, zbg_hit, judge, zbg_of_smiles
from s9_applicability import run_applicability

In [ ]:
warnings.filterwarnings('ignore')

# Output folder
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
(SPLIT_OUT, BASELINE_OUT, DMPNN_OUT, STACKING_OUT, COMPARISON_OUT, SCREENING_OUT, MODEL_OUT,
 APPLICABILITY_OUT) = (os.path.join(OUTPUT_DIR, n) for n in
    ('split', 'baseline', 'dmpnn', 'stacking', 'comparison', 'screening', 'model', 'applicability'))
for _d in (SPLIT_OUT, BASELINE_OUT, DMPNN_OUT, STACKING_OUT, COMPARISON_OUT, SCREENING_OUT, MODEL_OUT,
           APPLICABILITY_OUT):
    os.makedirs(_d, exist_ok=True)

# Device selection
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# Plotting configuration
plt.rcParams['figure.figsize'] = (13.33, 4)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['lines.markersize'] = 4
plt.rcParams['axes.grid'] = False
plt.rcParams['grid.alpha'] = 0.5
plt.rcParams['grid.linestyle'] = '-.'

In [ ]:
# Dataset 
df = pd.read_excel('data.xlsx')
print('shape:', df.shape)

# Clustering parameters (if the split is recomputed)
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.25
UMAP_N_COMPONENTS = 2
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 1

# Split train/val/test (same as the article)
SPLIT_DIR = 'split'
split_csv = [os.path.join(SPLIT_DIR, f'{s}_set_multitarget.csv') for s in ('train', 'val', 'test')]

if all(os.path.exists(f) for f in split_csv):
    train_df, val_df, test_df = (pd.read_csv(f) for f in split_csv)
    print(f'split loaded from {SPLIT_DIR}/ (reference partition of the published run)')
else:
    print(f'{SPLIT_DIR}/ not found: recomputing the split (UMAP + HDBSCAN)')
    train_df, val_df, test_df = splitter(df, output_dir=SPLIT_OUT,
                                         umap_n_neighbors=UMAP_N_NEIGHBORS,
                                         umap_min_dist=UMAP_MIN_DIST,
                                         umap_n_components=UMAP_N_COMPONENTS,
                                         hdbscan_min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
                                         hdbscan_min_samples=HDBSCAN_MIN_SAMPLES)

print(f'Train={len(train_df)} Val={len(val_df)} Test={len(test_df)}')
cv_df = pd.concat([train_df, val_df]).reset_index(drop=True)
print('target counts (train+val):', dict(zip(ISOFORMS, cv_df[TARGET_COLS].notna().sum().tolist())))

# Group sizes, largest first
group_sizes = cv_df[GROUP_COL].value_counts()
print(f'cv: {group_sizes.size} groups | largest cluster = {group_sizes.iloc[0]} '
      f'({100 * group_sizes.iloc[0] / len(cv_df):.1f}% of cv)')

In [ ]:
# UMAP parameters for this figure (different from the split's own UMAP)
VIZ_N_NEIGHBORS = 15
VIZ_MIN_DIST = 0.1
SEED = 42

# All molecules in one table, tagged with which split they belong to
split_df = pd.concat([train_df.assign(Split='Train'),
                      val_df.assign(Split='Val'),
                      test_df.assign(Split='Test')],
                     ignore_index=True)

# Morgan fingerprints of every molecule, keeping only the valid rows
fps, ok_idx = generate_fingerprints(split_df['Smiles'].astype(str).tolist())
split_df = split_df.iloc[ok_idx].reset_index(drop=True)
splits = split_df['Split'].values
n_mol = len(fps)
X_fp = fps_to_numpy(fps).astype(np.uint8)

# UMAP on ~7500 molecules takes about a minute
print(f'UMAP (Jaccard) on {n_mol} molecules...')
emb = umap.UMAP(n_components=2, n_neighbors=VIZ_N_NEIGHBORS, min_dist=VIZ_MIN_DIST,
                metric='jaccard', random_state=SEED, n_jobs=1).fit_transform(X_fp)

# Random split with the same sizes, as a point of comparison
n_te, n_va = int((splits == 'Test').sum()), int((splits == 'Val').sum())
perm = np.random.default_rng(SEED).permutation(n_mol)
splits_rnd = np.empty(n_mol, dtype=object)
splits_rnd[perm[:n_te]] = 'Test'
splits_rnd[perm[n_te:n_te + n_va]] = 'Val'
splits_rnd[perm[n_te + n_va:]] = 'Train'

# Median NN-similarity (test -> train+val) for each split
nn_clu, nn_rnd = nn_median(fps, splits), nn_median(fps, splits_rnd)

# Side-by-side UMAP panels: random split vs cluster-based split
fig, (axL, axR) = plt.subplots(1, 2, figsize=(18, 8.5), sharex=True, sharey=True, constrained_layout=True)
draw_split(axL, emb, splits_rnd, 'Random split')
draw_split(axR, emb, splits, 'Cluster-based split')
fig.savefig(os.path.join(SPLIT_OUT, 'umap_split_visualization.png'), dpi=300, bbox_inches='tight', transparent=True)
plt.show()

print(f'median NN-sim test->train | cluster-based={nn_clu:.3f}  random={nn_rnd:.3f}')

In [ ]:
# LightGBM baseline (fit on train+val, evaluated on test, OOF predictions for stacking)
lgbm_res = run_lgbm_baseline(train_df, val_df, test_df, BASELINE_OUT)

In [ ]:
# All molecules together
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
# Each molecule is its own group, so the baseline's internal CV is random here too
all_df[GROUP_COL] = np.arange(len(all_df))

# Reproducible random order
perm_rnd = np.random.default_rng(42).permutation(len(all_df))
# Same test/val sizes as the cluster-based split
n_te, n_va = len(test_df), len(val_df)
rnd_test = all_df.iloc[perm_rnd[:n_te]].reset_index(drop=True)
rnd_val = all_df.iloc[perm_rnd[n_te:n_te + n_va]].reset_index(drop=True)
rnd_train = all_df.iloc[perm_rnd[n_te + n_va:]].reset_index(drop=True)
print(f'random split: Train={len(rnd_train)} Val={len(rnd_val)} Test={len(rnd_test)} '
      f'(same sizes as the cluster-based split)')

# Same baseline, but on the random split (results in a subfolder)
lgbm_rnd = run_lgbm_baseline(rnd_train, rnd_val, rnd_test, os.path.join(SPLIT_OUT, 'random_split_baseline'))

# Comparison table between the two splits
cmp_df = pd.DataFrame([
    {'split': 'random', 'NN_sim_median': nn_rnd,
     'R2_pooled': lgbm_rnd['test_metrics']['r2'], 'MSE': lgbm_rnd['test_metrics']['mse'],
     'Spearman': lgbm_rnd['test_metrics']['spearman_r']},
    {'split': 'cluster-based', 'NN_sim_median': nn_clu,
     'R2_pooled': lgbm_res['test_metrics']['r2'], 'MSE': lgbm_res['test_metrics']['mse'],
     'Spearman': lgbm_res['test_metrics']['spearman_r']},
])
cmp_df.to_csv(os.path.join(SPLIT_OUT, 'split_difficulty.csv'), index=False)
display(cmp_df.round(3))
# How much the random split "inflates" R2
delta = lgbm_rnd['test_metrics']['r2'] - lgbm_res['test_metrics']['r2']
print(f'\nR2 inflated by the random split: {delta:+.3f}  (median NN-sim {nn_rnd:.2f} against {nn_clu:.2f})')

# Bar chart of the two R2 values
fig, ax = plt.subplots(figsize=(7, 4.5))
vals = [lgbm_rnd['test_metrics']['r2'], lgbm_res['test_metrics']['r2']]
ax.bar(['random', 'cluster-based'], vals, color=['#bdbdbd', '#ed6a5a'], edgecolor='black', width=0.55)
for i, v in enumerate(vals):
    ax.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=13)
ax.set_ylabel('Test R2 pooled (baseline LightGBM)')
ax.set_title('Same model, two splitting criteria')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(SPLIT_OUT, 'split_difficulty.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Number of seeds: full training is repeated N_SEEDS times with different initializations
N_SEEDS = 2

# D-MPNN architecture hyperparameters
DMPNN_KWARGS = dict(hidden_dim=160, num_layers=4, conv_dropout=0.05, set2set_steps=3,
                    fc1_hidden=192, fc2_hidden=96, fc_dropout=0.3)

# Training hyperparameters
TRAIN_KWARGS = dict(k_folds=5, epochs=120, patience=18, batch_size=32, lr=5e-4, weight_decay=0.01,
                    grad_clip=1.0,
                    edge_drop_prob=0.10,  # augmentation: probability of dropping a bond
                    node_mask_prob=0.05,  # augmentation: probability of zeroing an atom
                    swa_start_frac=0.75,  # SWA starts at 75% of epochs (ep. 90)
                    swa_lr=1e-4,
                    fold_n_trials=30,  # attempts to pick the most balanced folds
                    fold_seed=42,  # fold seed, shared by every training seed
                    num_workers=4)

# True pIC50 of the test set (molecules x 9)
test_y = test_df[TARGET_COLS].values.astype(np.float32)
# True pIC50 of train+val, in the same order as the OOF predictions
Y_cv = cv_df[TARGET_COLS].values.astype(np.float32)
# Baseline test R2, as a reference point
LGBM_TEST_R2 = lgbm_res['test_metrics']['r2']
print(f"[config] N_SEEDS={N_SEEDS}  EPOCHS={TRAIN_KWARGS['epochs']}  K_FOLDS={TRAIN_KWARGS['k_folds']}  "
      f"|  LGBM baseline R2={LGBM_TEST_R2:.4f}")

In [ ]:
# Step 5: cross-validated training for every seed (takes hours on GPU)
res_all = train_dmpnn(cv_df, N_SEEDS, DMPNN_KWARGS, 'pretrained/ssl_encoder_dmpnn.pt', DMPNN_OUT, device,
                      **TRAIN_KWARGS)

# The seed is chosen on cross-validation (OOF R²) only; the test set is just evaluated
best_seed, seed_cv_r2 = select_best_seed(res_all, Y_cv)
# Test predictions of every seed, to show how much they vary
seed_test_preds = [predict_dmpnn(seed_view(res_all, s), test_df['Smiles'].tolist(), device,
                                 batch_size=TRAIN_KWARGS['batch_size']) for s in range(N_SEEDS)]
seed_test_r2 = [calculate_metrics(p, test_y)['r2'] for p in seed_test_preds]

# Chosen seed: every later cell uses only this one
res_dmpnn = seed_view(res_all, best_seed)
dmpnn_test_preds = seed_test_preds[best_seed]
m_te_dmpnn = calculate_metrics(dmpnn_test_preds, test_y)

print(f'\nD-MPNN  seed {best_seed}, chosen on CV (OOF R² {seed_cv_r2[best_seed]:.3f})')
print(f'  test  R² {m_te_dmpnn["r2"]:.3f}  MSE {m_te_dmpnn["mse"]:.3f}  '
      f'Spearman {m_te_dmpnn["spearman_r"]:.3f}   (LightGBM R² {LGBM_TEST_R2:.3f})')
if N_SEEDS > 1:
    others = '  '.join(f'seed {s} {r:.3f}' for s, r in enumerate(seed_test_r2) if s != best_seed)
    print(f'  other seeds on test: {others}  ->  spread Δ {max(seed_test_r2) - min(seed_test_r2):.3f}')

# Save each seed's CV and test R2
pd.DataFrame({'seed': range(N_SEEDS), 'cv_oof_r2': seed_cv_r2, 'test_r2': seed_test_r2}).to_csv(
    os.path.join(DMPNN_OUT, 'seed_summary.csv'), index=False)
# Per-isoform table and parity plots of the chosen seed
per_isoform_report(test_y, dmpnn_test_preds, output_path=os.path.join(DMPNN_OUT, 'per_isoform_test.csv'))
parity_plots(test_y, dmpnn_test_preds, output_path=os.path.join(DMPNN_OUT, 'parity_test.png'))
display(Image(filename=os.path.join(DMPNN_OUT, 'per_isoform_test.png')))
display(Image(filename=os.path.join(DMPNN_OUT, 'parity_test.png')))

In [ ]:
# Ridge penalty values stacking chooses from (inner CV)
ALPHA_GRID = [0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]

# Out-of-fold predictions of both models on train+val: stacking's training data
oof_dict = {'LGBM': lgbm_res['oof'], 'DMPNN': res_dmpnn['oof']}
test_dict = {'LGBM': lgbm_res['test_preds'], 'DMPNN': dmpnn_test_preds}
# One Ridge per isoform, fit on the OOF predictions
meta = fit_stacking_meta_learner(oof_dict, Y_cv, alpha_grid=ALPHA_GRID, cv_splits=5)
stacked_preds = predict_stacking(meta, test_dict)
stack_metrics = calculate_metrics(stacked_preds, test_y)
print(f"[stacking LGBM+DMPNN, weights>=0]  Test R2={stack_metrics['r2']:.4f}  "
      f"MSE={stack_metrics['mse']:.4f}  Spearman={stack_metrics['spearman_r']:.4f}")

per_isoform_report(test_y, stacked_preds, output_path=os.path.join(STACKING_OUT, 'per_isoform_test.csv'))
parity_plots(test_y, stacked_preds, output_path=os.path.join(STACKING_OUT, 'parity_test.png'))
display(Image(filename=os.path.join(STACKING_OUT, 'per_isoform_test.png')))
display(Image(filename=os.path.join(STACKING_OUT, 'parity_test.png')))

In [ ]:
# Test predictions of the three models being compared
all_test_preds = {'LGBM': lgbm_res['test_preds'],  # baseline
                  'DMPNN': dmpnn_test_preds,  # D-MPNN, chosen seed
                  'Stacking': stacked_preds}  # combination of the two

# Per-isoform tables (regression + ranking)
per_iso_full = {}
for name, preds in all_test_preds.items():
    rnk = ranking_metrics_per_isoform(test_y, preds,
                                      output_path=os.path.join(COMPARISON_OUT, f'ranking_{name.lower()}.csv'))
    t = full_per_isoform_table(test_y, preds, ranking_df=rnk)
    per_iso_full[name] = t
    t.round(4).to_csv(os.path.join(COMPARISON_OUT, f'per_isoform_{name.lower()}.csv'), index=False)
    print(f'\n================  PER-ISOFORM (test) - {name}  ================')
    display(t.round(3))

# Summary table
consolidated = pd.DataFrame({'isoform': ISOFORMS, 'n_test': per_iso_full['Stacking']['n'].values[:len(ISOFORMS)]})
for name in all_test_preds:
    src = per_iso_full[name].iloc[:len(ISOFORMS)]
    consolidated[f'R2_{name}'] = src['R2'].values
    consolidated[f'MAE_{name}'] = src['MAE'].values
    consolidated[f'Spearman_{name}'] = src['Spearman'].values
consolidated.to_csv(os.path.join(COMPARISON_OUT, 'per_isoform_consolidated.csv'), index=False)

# R2 per isoform
r2_comparison_plot(consolidated, os.path.join(COMPARISON_OUT, 'r2_per_isoform_comparison.png'),
                   underpowered_n=UNDERPOWERED_N)
display(Image(filename=os.path.join(COMPARISON_OUT, 'r2_per_isoform_comparison.png')))

# Test molecules per isoform
n_test = consolidated['n_test'].values
underpowered = [iso for iso, n in zip(ISOFORMS, n_test) if n < UNDERPOWERED_N]
print(f'\nUNDERPOWERED isoforms (n<{UNDERPOWERED_N}, R2 not to be over-interpreted): {underpowered}')
print('Macro summary (mean over isoforms)  [all | n>=25 only]:')
for name in all_test_preds:
    t = per_iso_full[name].iloc[:len(ISOFORMS)]
    powered = t[t['n'] >= UNDERPOWERED_N]
    print(f'  {name:9s}  R2_all={np.nanmean(t["R2"]):.4f}  R2_pow={np.nanmean(powered["R2"]):.4f}  '
          f'Spearman={np.nanmean(t["Spearman"]):.4f}  EF@10%={np.nanmean(t["EF@10%"]):.4f}')

In [ ]:
# D-MPNN cross-validation R2 (chosen seed)
cv_r2 = np.array([m['r2'] for m in res_dmpnn['fold_metrics']], dtype=float)

# One row per model
rows = [
    {'model': 'LightGBM (Morgan FP) [baseline]', 'r2_cv_mean': np.nan, 'r2_cv_std': np.nan,
     'r2_test': lgbm_res['test_metrics']['r2'], 'mse_test': lgbm_res['test_metrics']['mse'],
     'spearman_test': lgbm_res['test_metrics']['spearman_r']},
    {'model': f'D-MPNN (seed {best_seed}, tgt_std+pretrain, uncertainty)', 'r2_cv_mean': cv_r2.mean(),
     'r2_cv_std': cv_r2.std(), 'r2_test': m_te_dmpnn['r2'], 'mse_test': m_te_dmpnn['mse'],
     'spearman_test': m_te_dmpnn['spearman_r']},
    {'model': 'Stacking LightGBM + D-MPNN', 'r2_cv_mean': np.nan, 'r2_cv_std': np.nan,
     'r2_test': stack_metrics['r2'], 'mse_test': stack_metrics['mse'], 'spearman_test': stack_metrics['spearman_r']},
]
final_df = pd.DataFrame(rows)
final_df.to_csv(os.path.join(COMPARISON_OUT, 'final_comparison.csv'), index=False)
final_df

In [ ]:
# Bundle the chosen seed's D-MPNN, LightGBM and stacking into one file
bundle_path = os.path.join(MODEL_OUT, 'inference_bundle.pt')
export_bundle(bundle_path, res_dmpnn, meta, lgbm_res['models'])
# Reload it, as an external user would
bundle = load_bundle(bundle_path, map_location=device)
PRED_COLS = bundle['target_cols']

# Re-read the full dataset and predict all 9 isoforms for every molecule
df_full = pd.read_excel('data.xlsx')
preds_all = predict(df_full['Smiles'].astype(str).tolist(), bundle, device=device)
for j, col in enumerate(TARGET_COLS):
    # Fill only the unmeasured cells (imputation); real measurements are untouched
    vals = df_full[col].to_numpy(dtype=float, copy=True)
    nan_mask = np.isnan(vals)
    vals[nan_mask] = preds_all[nan_mask, j]
    df_full[col] = vals

# Class I selectivity score for every molecule
df_full['score'] = selectivity_score(df_full)[2]
df_full.to_excel(os.path.join(SCREENING_OUT, 'dataset_scored.xlsx'), index=False)

In [ ]:
# External control library
BITTER_CSV = 'bitter.csv'
BITTER_SMILES_COL = 'canonical_smiles'

bitter = pd.read_csv(BITTER_CSV)
print(f'bitter.csv: {len(bitter)} rows | SMILES column = "{BITTER_SMILES_COL}"')

# Clean SMILES: largest fragment only (strips salts/counterions)
bitter['smiles_clean'] = bitter[BITTER_SMILES_COL].map(largest_fragment)
n_bad = int(bitter['smiles_clean'].isna().sum())
bitter = bitter.dropna(subset=['smiles_clean']).reset_index(drop=True)
print(f'unparsed SMILES discarded: {n_bad} | valid molecules: {len(bitter)}')

# Full-chain predictions and selectivity score; predictions themselves aren't saved
pred_df = predict_to_dataframe(bitter['smiles_clean'].tolist(), bundle, device=device)
bitter['mean_class_I'], bitter['max_others'], bitter['score'] = selectivity_score(pred_df)
bitter = (bitter[['smiles_clean', 'mean_class_I', 'max_others', 'score']]
          .sort_values('score', ascending=False).reset_index(drop=True))

out_path = os.path.join(SCREENING_OUT, 'bitter_scores.csv')
bitter.round(4).to_csv(out_path, index=False)
print(f'saved -> {out_path}  ({len(bitter)} molecules)')
print(f"selectivity score: mean={bitter['score'].mean():.2f}  max={bitter['score'].max():.2f}  "
      f"min={bitter['score'].min():.2f}  | compounds with score>0: {(bitter['score'] > 0).sum()}")

In [ ]:
# Full dataset sorted by selectivity, most selective first
df_full = df_full.sort_values('score', ascending=False).reset_index(drop=True)
save_top_table(df_full, SMILES_COL, os.path.join(SCREENING_OUT, 'dataset_top15.png'), 'Top 15 - df_full')

# Scores of both libraries (NaN dropped)
s_full = df_full['score'].dropna().to_numpy()  # full dataset (the focus)
s_bitter = bitter['score'].dropna().to_numpy()  # external reference library

# Shared x-axis range (0.5-99.5 percentile, so outliers don't squash the plot) and bins
lo, hi = np.percentile(np.concatenate([s_full, s_bitter]), [0.5, 99.5])
bins = np.linspace(lo, hi, 41)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12.5, 4.8), sharex=True, facecolor=SURFACE,
                               gridspec_kw={'wspace': 0.22})
for ax in (axL, axR):
    ax.set_facecolor(SURFACE)
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.spines['bottom'].set_color(GRID)
    ax.grid(axis='y', color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK_SOFT, length=0, labelsize=11)
    ax.axvspan(0, hi, color=C_BLUE, alpha=0.05, lw=0)  # "Class I selective" zone (score > 0)
    ax.axvline(0, color=INK_SOFT, lw=1, ls=':')

# Left: filled histogram for the dataset (focus), outline only for the external library
axL.hist(s_full, bins=bins, density=True, color=C_BLUE, alpha=0.85,
         edgecolor=SURFACE, linewidth=0.8, label=f'Dataset (n={len(s_full):,})', zorder=3)
axL.hist(s_bitter, bins=bins, density=True, histtype='step', color=C_ORANGE,
         linewidth=2, label=f'Bitter (n={len(s_bitter):,})', zorder=4)
# Medians, dashed in each series' colour
for s, c, side in [(s_full, C_BLUE, -1), (s_bitter, C_ORANGE, 1)]:
    med = np.median(s)
    axL.axvline(med, color=c, ls='--', lw=1.6, zorder=5)
    axL.annotate(f'median {med:.2f}', (med, 1.0), xycoords=('data', 'axes fraction'), xytext=(7 * side, 6),
                 textcoords='offset points', va='bottom', ha='left' if side > 0 else 'right',
                 fontsize=10.5, color=INK_SOFT, annotation_clip=False)
axL.set_ylabel('Density', color=INK_SOFT, fontsize=12)
axL.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 0.86), fontsize=11, labelcolor=INK)

# Right: cumulative distributions, thin step lines
for s, c in [(s_full, C_BLUE), (s_bitter, C_ORANGE)]:
    xs, ys = ecdf(s)
    axR.step(xs, ys, where='post', color=c, lw=2.2, zorder=3)
axR.annotate(f'{100 * (s_full > 0).mean():.0f}% of dataset\nhas score > 0', (0, np.mean(s_full <= 0)),
             xytext=(14, -6), textcoords='offset points', va='top', fontsize=10.5, color=INK_SOFT,
             arrowprops=dict(arrowstyle='-', color=INK_SOFT, lw=0.8))
axR.set_ylim(0, 1)
axR.set_ylabel('Cumulative fraction', color=INK_SOFT, fontsize=12)

axL.set_xlim(lo, hi)
fig.supxlabel('Class I selectivity score  (mean Class I pIC50 − max other isoform)', color=INK_SOFT, fontsize=12)
fig.suptitle('Selectivity score: dataset vs bitter compounds', x=0.07, ha='left', fontsize=15,
             weight='bold', color=INK)
plt.savefig(os.path.join(SCREENING_OUT, 'score_comparison.png'), bbox_inches='tight', dpi=200, facecolor=fig.get_facecolor())
plt.show()

print('--- Class I selectivity score ---')
summ('df_full', s_full)
summ('bitter', s_bitter)

In [ ]:
# Full dataset ranked by score, with each molecule's known ZBG (if any)
work = df_full.sort_values('score', ascending=False).copy()
work['ZBG'] = work['Smiles'].map(zbg_of_smiles)

n_lib, n_hit = len(work), int(work['ZBG'].notna().sum())
print(f'df_full: {n_hit}/{n_lib} molecules ({100 * n_hit / n_lib:.1f}%) carry a known group')
print(f'among the top-20 by score: {int(work.head(20)["ZBG"].notna().sum())}/20 carry a known group')

# Top 20 by score that also carry a known ZBG
hit = work[work['ZBG'].notna()].head(20).reset_index(drop=True)
hit['mean_class_I'], hit['max_others'], _ = selectivity_score(hit)
print(f'\nshowing {len(hit)} molecules (top by score carrying a known group)')
print('group distribution among the selected molecules:')
print(hit['ZBG'].value_counts().to_string())

# Structures with the ZBG highlighted, plus a table of score and drug-likeness
mols, legends, highlights, rows = [], [], [], []
for i, r in hit.iterrows():
    m = Chem.MolFromSmiles(str(r['Smiles']))
    name, atoms = zbg_hit(m)
    mols.append(m)
    highlights.append(atoms)
    legends.append(f"#{i + 1}  s={r['score']:.2f}  {name}")
    rows.append({'rank': i + 1, 'score': round(r['score'], 2), 'mean_class_I': round(r['mean_class_I'], 2),
                 'max_others': round(r['max_others'], 2), 'ZBG': name, **judge(m), 'SMILES': str(r['Smiles'])[:55]})

img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(260, 220), legends=legends,
                           highlightAtomLists=highlights, useSVG=False)
display(img)

display(pd.DataFrame(rows))

In [ ]:
# Step 9: applicability domain
# Whether a prediction can be trusted is decided by measured error, not assumed: out-of-fold and
# test predictions are binned by nearest-neighbour similarity to the training set, and the domain
# threshold is placed where the RMSE in a bin first exceeds one pIC50 unit (a tenfold IC50 error).
oof_path = os.path.join(DMPNN_OUT, f'seed{best_seed}_oof.npz')
ad_res = run_applicability(cv_df, test_df, oof_path, bundle, bitter['smiles_clean'].tolist(),
                           APPLICABILITY_OUT, device=device)

display(ad_res['calibration'][['bin', 'n_mol', 'n_obs', 'mean_sim', 'rmse', 'r2', 'spearman_r']].round(3))

# The BitterDB screen, re-read through the domain filter
ad = ad_res['annotated']
class_I_pred = [f'pred_{iso}_HUMAN' for iso in CLASS_I_ISOFORMS]
other_pred = [f'pred_{c}' for c in TARGET_COLS if f'pred_{c}' not in class_I_pred]
ad['mean_class_I'] = ad[class_I_pred].mean(axis=1)
ad['max_others'] = ad[other_pred].max(axis=1)
ad['score'] = ad['mean_class_I'] - ad['max_others']

active = ad['mean_class_I'] > 6  # predicted better than 1 uM on Class I
print()
print(f"predicted above 1 uM: {int(active.sum())}  |  of those inside the domain: "
      f"{int((active & ad['in_domain']).sum())}")
top15 = ad.nlargest(15, 'score')
print(f"unfiltered top-15 by score: {int(top15['in_domain'].sum())}/15 inside the domain, "
      f"median NN-Tanimoto {top15['nn_tanimoto'].median():.3f}")

# Only the compounds the model is entitled to rank
print()
print('best in-domain compounds by selectivity score:')
display(ad[ad['in_domain']].nlargest(10, 'score')[
    ['Smiles', 'nn_tanimoto', 'mean_class_I', 'max_others', 'score', 'pred_clipped']].round(3))